# B4 - Convolutional Autoencoder on MNIST
Encoder: Conv32 -> Pool -> Conv16 -> Pool -> Conv8 (latent 7x7x8)

Decoder: Conv8 -> Up -> Conv16 -> Up -> Conv32 -> Conv1 sigmoid

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt

In [ ]:
# 1. Load MNIST, normalize to [0, 1]
(x_train, _), (x_test, _) = keras.datasets.mnist.load_data()
x_train = (x_train.astype("float32") / 255.0)[..., None]
x_test = (x_test.astype("float32") / 255.0)[..., None]
print(x_train.shape, x_test.shape)

In [ ]:
# 2. Encoder and decoder
class ConvAutoencoder(keras.Model):
    def __init__(self):
        super().__init__()
        self.encoder = keras.Sequential([
            layers.Input((28, 28, 1)),
            layers.Conv2D(32, 3, activation='relu', padding='same'),   # 28x28x32
            layers.MaxPooling2D(2),                                    # 14x14x32
            layers.Conv2D(16, 3, activation='relu', padding='same'),   # 14x14x16
            layers.MaxPooling2D(2),                                    # 7x7x16
            layers.Conv2D(8, 3, activation='relu', padding='same'),    # 7x7x8 latent
        ], name='encoder')
        self.decoder = keras.Sequential([
            layers.Input((7, 7, 8)),
            layers.Conv2D(8, 3, activation='relu', padding='same'),    # 7x7x8
            layers.UpSampling2D(2),                                    # 14x14x8
            layers.Conv2D(16, 3, activation='relu', padding='same'),   # 14x14x16
            layers.UpSampling2D(2),                                    # 28x28x16
            layers.Conv2D(32, 3, activation='relu', padding='same'),   # 28x28x32
            layers.Conv2D(1, 3, activation='sigmoid', padding='same'), # 28x28x1
        ], name='decoder')

    def call(self, x):
        return self.decoder(self.encoder(x))

model = ConvAutoencoder()
model.encoder.summary(); model.decoder.summary()

In [ ]:
# 3. Train (BCE loss, 8 epochs)
model.compile(optimizer='adam', loss='binary_crossentropy')
history = model.fit(x_train, x_train, epochs=8, batch_size=128, validation_data=(x_test, x_test))

In [ ]:
# 4. Plot loss, evaluate, show reconstructions
plt.plot(history.history['loss'], label='train'); plt.plot(history.history['val_loss'], label='val')
plt.xlabel('epoch'); plt.ylabel('BCE loss'); plt.legend(); plt.title('Training loss'); plt.show()

print("Test loss:", model.evaluate(x_test, x_test, verbose=0))

recon = model.predict(x_test[:10])
plt.figure(figsize=(12, 3))
for i in range(10):
    plt.subplot(2, 10, i + 1); plt.imshow(x_test[i, :, :, 0], cmap='gray'); plt.axis('off')
    plt.subplot(2, 10, 10 + i + 1); plt.imshow(recon[i, :, :, 0], cmap='gray'); plt.axis('off')
plt.suptitle('Top: original   Bottom: reconstructed'); plt.show()